In [137]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [138]:
df = pd.read_csv("/kaggle/input/datasets/hridoybanik/churn-customer-svm/WA_Fn-UseC_-Telco-Customer-Churn (1).csv")

In [139]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [140]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [141]:
df = df.dropna(subset=['TotalCharges'])

In [142]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [143]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [144]:
df['TotalCharges'].value_counts()

TotalCharges
20.20      11
19.75       9
19.65       8
19.90       8
20.05       8
           ..
130.15      1
3211.90     1
7843.55     1
2196.30     1
197.40      1
Name: count, Length: 6530, dtype: int64

In [145]:
df.drop('customerID', axis =1, inplace = True)

In [146]:
from sklearn.compose import make_column_selector as selector

# This automatically creates a function that grabs object and category dtypes
cat_selector = selector(dtype_include=['object', 'category'])
categorical_cols = cat_selector(df)

print("Pipeline Categorical Columns:", categorical_cols)

Pipeline Categorical Columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


In [147]:
import numpy as np
from sklearn.compose import make_column_selector as selector

# Automatically creates a selector that grabs all numeric dtypes (integers and floats)
num_selector = selector(dtype_include=['number'])  # Or use: dtype_include=[np.number]
numerical_cols = num_selector(df)

print("Pipeline Numerical Columns:", numerical_cols)

Pipeline Numerical Columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [148]:
cat_col = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
num_col =  ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [149]:
X = df.drop("Churn", axis =1)
y = df['Churn']

In [150]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


In [151]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn import svm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

In [152]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)


In [153]:
p1 = Pipeline(
    steps = [
        ('Scaling_numerical data', StandardScaler())
        
    ]
)
p2 = Pipeline(
    steps = [
        ('OHE', OneHotEncoder(drop='first',sparse_output=False))
        
    ]
)


In [154]:
preprocessing = ColumnTransformer(
    transformers = [
        ("Preprocessor_num", p1, num_col),
        ("Preprocessor_cat", p2, cat_col)
    ], remainder = 'passthrough'
)

In [155]:
model_trainings = Pipeline(
    steps = [
        ("process", preprocessing),
        ("model_training", svm.SVC())
    ]
)

In [156]:
model_training.fit(X_train,y_train)

Pipeline(steps=[('process',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Preprocessor_num',
                                                  Pipeline(steps=[('Scaling_numerical '
                                                                   'data',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('Preprocessor_cat',
                                                  Pipeline(steps=[('OHE',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('model_training', SVC(kernel='linear'))])

In [157]:
y_pred = model_training.predict(X_test)
from sklearn.metrics import accuracy_score, precision_score, f1_score
print("For linear Kernel")
print("Testing Accuracy")
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score:  {f1:.4f}")


For linear Kernel
Testing Accuracy
Accuracy:  0.7946
Precision: 0.6349
F1 Score:  0.5806


In [167]:
param_grid = [
    {
        "model_training__kernel": ["rbf"],
        "model_training__C" : [0.01 , 0.1 , 1 , 10 , 50 , 100],
        "model_training__gamma" : ['scale', 'auto']
    },
    {
        "model_training__kernel" : ["poly"],
        "model_training__C" : [0.01 , 0.1 , 1 , 10 , 50 , 100],
        "model_training__degree": [1,2,3,4,5,6,7,8],
        "model_training__gamma" : ['scale', 'auto']
    },
    {
        "model_training__kernel" : ["linear"],
        "model_training__C" : [0.01 , 0.1 , 1 , 10 , 50 , 100] 
    },
    {
        "model_training__kernel" : ['sigmoid'],
        "model_training__C" : [0.01 , 0.1 , 1 , 10 , 50 , 100],
        "model_training__gamma" : ['scale', 'auto']
    }
 
]

In [168]:
grid_search = GridSearchCV(
    estimator = model_training, 
    param_grid = param_grid,
    cv = 5,
    n_jobs = -1
)

In [169]:
grid_search.fit(X_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('process',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('Preprocessor_num',
                                                                         Pipeline(steps=[('Scaling_numerical '
                                                                                          'data',
                                                                                          StandardScaler())]),
                                                                         ['SeniorCitizen',
                                                                          'tenure',
                                                                          'MonthlyCharges',
                                                                          'TotalCharges']),
                                                                        ('Preprocessor_cat',
                                                                         Pipeline(steps=[('OHE',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        sparse_output=False))]),
                                                                         ['...
                         {'model_training__C': [0.01, 0.1, 1, 10, 50, 100],
                          'model_training__degree': [1, 2, 3, 4, 5, 6, 7, 8],
                          'model_training__gamma': ['scale', 'auto'],
                          'model_training__kernel': ['poly']},
                         {'model_training__C': [0.01, 0.1, 1, 10, 50, 100],
                          'model_training__kernel': ['linear']},
                         {'model_training__C': [0.01, 0.1, 1, 10, 50, 100],
                          'model_training__gamma': ['scale', 'auto'],
                          'model_training__kernel': ['sigmoid']}])

In [170]:
y_pred_best = grid_search.predict(X_test)

In [171]:
from sklearn.metrics import accuracy_score, precision_score, f1_score
print("For Grid Search CV")
print("Testing Accuracy")
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_best)
precision = precision_score(y_test, y_pred_best)
f1 = f1_score(y_test, y_pred_best)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score:  {f1:.4f}")

For Grid Search CV
Testing Accuracy
Accuracy:  0.7932
Precision: 0.6653
F1 Score:  0.5344
